In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [3]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='ollama:gemma4:e4b',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='ollama:gemma4:e4b',
    tools=[square]
)

## Calling subagents

In [4]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='ollama:gemma4:e4b',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [6]:
question = "What is the square of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square of 456?', additional_kwargs={}, response_metadata={}, id='392a26a7-1235-4c6e-a3f8-f8ad52c1b7a5'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-07T10:34:51.983897Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4549139083, 'load_duration': 188686916, 'prompt_eval_count': 155, 'prompt_eval_duration': 3783680666, 'eval_count': 18, 'eval_duration': 565308083, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d6782-5307-7e80-b9c7-3480245a3c3c-0', tool_calls=[{'name': 'call_subagent_2', 'args': {'x': 456}, 'id': 'ceade12e-12bd-48f9-b7bf-ac7d098e07e0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 155, 'output_tokens': 18, 'total_tokens': 173}),
              ToolMessage(content='The square of 456.0 is 207936.0.', name='call_subagent_2', id='4af9026b-4152-4a54-af60-5c5922a485d5', t